# Sentinel · reranker v3: distil the baseline into a 2-layer student (speed)

Reranking 50 candidates takes ~1 s on CPU — as long as the LLM call. v3 keeps the baseline
(MiniLM-L6) as the **teacher** and trains a 2-layer **student** to reproduce its rankings on the
RFC candidate pools: ~3x less compute per pair.

1. **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all.** About 40 minutes. The last cell downloads `sentinel-reranker-v3.zip` — put it in Downloads.

**Fixed before any result (pre-registered):**
- teacher: the baseline itself, `cross-encoder/ms-marco-MiniLM-L-6-v2@233902d`, raw scores on every (question, candidate) pair
- two students, both 2 layers: **A** = public `cross-encoder/ms-marco-MiniLM-L2-v2` (pinned) · **B** = the baseline cut to the 2 layers that agree best with the teacher on 300 validation questions, untrained
- pure listwise distillation (KL, T = 1, weight 1, no label filter), lr 5e-5, 3 epochs, group 1 + 15, batch 16, warmup 10 %, validate every 300 steps on 800 held-out questions
- checkpoint and arm chosen by validation agreement with the teacher (`teacher_mrr@10`) — the ground truth is never read here
- ship rule, applied once afterwards on the 48 ground-truth questions: ≥ 2x faster and MRR@10 within 0.02 of the baseline

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
!test -d Sentinel || git clone --depth 1 --branch feat/reranker-finetune https://github.com/krishivsaini/Sentinel.git
%cd Sentinel
!git log --oneline -1

In [ ]:
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

**1. Teacher scores** — the baseline scores all ~441k (question, candidate) pairs. Resumable.

In [ ]:
!python scripts/teacher_scores.py --model cross-encoder/ms-marco-MiniLM-L-6-v2 --revision 233902d25c440f23af6f7d6e94d2946bac0bee0a --out data/reranker/teacher-l6.jsonl.gz --device cuda --batch-pairs 256

In [ ]:
import gzip, json, statistics
rows = [json.loads(l) for l in gzip.open('data/reranker/teacher-l6.jsonl.gz', 'rt')]
ranks = [1 + sum(x > r['scores'][0] for x in r['scores'][1:]) for r in rows]
print(len(rows), 'questions | baseline ranks the labelled positive 1st in %.0f%% (v1 diagnosis: ~40%%)' % (100 * sum(r == 1 for r in ranks) / len(ranks)))

**2. Student B's layers** — every 2-of-6 choice, untrained, scored on 300 validation questions.

In [ ]:
!python scripts/pick_student_layers.py --teacher-path data/reranker/teacher-l6.jsonl.gz --keep 2 --device cuda --out runs/v3-student-layers.json
LAYERS = ','.join(map(str, json.load(open('runs/v3-student-layers.json'))['best']))
print('student B keeps layers', LAYERS)

**3. Check the loop on the GPU** — seconds.

In [ ]:
!python scripts/train_reranker.py --smoke --device cuda --teacher-path data/reranker/teacher-l6.jsonl.gz --teacher-max-rank 0 --distill-weight 1 --student-layers 0,5 --select-metric teacher_mrr@10 --out-dir runs/v3-smoke

**4. Student A** — public MiniLM-L2, distilled from the baseline on RFC pools.

In [ ]:
!python scripts/train_reranker.py --base-model cross-encoder/ms-marco-MiniLM-L2-v2 --base-revision 1b5cd67b15209f24824c50370e0397743aa9b787 --teacher-path data/reranker/teacher-l6.jsonl.gz --teacher-max-rank 0 --distill-weight 1 --select-metric teacher_mrr@10 --negatives 15 --lr 5e-5 --epochs 3 --eval-every 300 --limit-val 800 --patience 2 --out-dir runs/v3-l2

**5. Student B** — the baseline itself, cut to 2 layers, distilled.

In [ ]:
!python scripts/train_reranker.py --student-layers $LAYERS --teacher-path data/reranker/teacher-l6.jsonl.gz --teacher-max-rank 0 --distill-weight 1 --select-metric teacher_mrr@10 --negatives 15 --lr 5e-5 --epochs 3 --eval-every 300 --limit-val 800 --patience 2 --out-dir runs/v3-cut

**6. Pick the arm** — by validation agreement with the teacher only.

In [ ]:
arms = {a: json.load(open(f'runs/{a}/summary.json')) for a in ['v3-l2', 'v3-cut']}
for a, s in arms.items():
    v = s['best_val']
    print(f"{a:7} teacher_mrr@10 {v['teacher_mrr@10']:.4f}  teacher_top5 {v['teacher_top5']:.4f}  label mrr@10 {v['mrr@10']:.4f}  best step {s['best_step']}  {s['seconds']:.0f}s")
chosen = max(arms, key=lambda a: arms[a]['best_val']['teacher_mrr@10'])
json.dump({'chosen': chosen, 'rule': 'max validation teacher_mrr@10',
           'arms': {a: s['best_val'] for a, s in arms.items()}}, open('runs/v3-selection.json', 'w'), indent=2)
print('chosen:', chosen)

**7. CPU latency on x86** (Colab's CPU) — 50 candidates per question, fp32 and dynamic int8 (fbgemm). Relative numbers; the Mac is measured locally.

In [ ]:
import copy, gzip, json, random, time
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer

chunk = {r['chunk_id']: r['text'] for r in map(json.loads, gzip.open('data/reranker/chunks.jsonl.gz', 'rt'))}
val = [json.loads(l) for l in gzip.open('data/reranker/val.jsonl.gz', 'rt')]
random.Random(0).shuffle(val)
batches = []                                   # 50 pairs: a question's 31 candidates + 19 more
for a, b in zip(val[:30], val[30:60]):
    cands = [a['positive'], *a['negatives'], b['positive'], *b['negatives']][:50]
    batches.append([(a['query'], chunk[c]) for c in cands])

def timed(model, tok, reps=1):
    model.eval(); out = []
    with torch.inference_mode():
        for pairs in batches[:3]:              # warm-up
            enc = tok(*map(list, zip(*pairs)), truncation=True, max_length=512, padding=True, return_tensors='pt'); model(**enc)
        for pairs in batches * reps:
            t = time.perf_counter()
            for s in range(0, 50, 32):        # CrossEncoder.predict's default batch size
                p = pairs[s:s + 32]
                enc = tok([q for q, _ in p], [d for _, d in p], truncation=True, max_length=512, padding=True, return_tensors='pt')
                model(**enc)
            out.append((time.perf_counter() - t) * 1000)
    return round(float(np.median(out)), 1)

models = {'baseline': ('cross-encoder/ms-marco-MiniLM-L-6-v2', '233902d25c440f23af6f7d6e94d2946bac0bee0a'), 'student': (f'runs/{chosen}/best', None)}
lat = {'cpu_threads': torch.get_num_threads(), 'quantized_engine': torch.backends.quantized.engine, 'p50_ms_pool50': {}}
for name, (path, rev) in models.items():
    tok = AutoTokenizer.from_pretrained(path, revision=rev)
    fp32 = AutoModelForSequenceClassification.from_pretrained(path, revision=rev).to('cpu')
    int8 = torch.ao.quantization.quantize_dynamic(copy.deepcopy(fp32), {torch.nn.Linear}, dtype=torch.qint8)
    lat['p50_ms_pool50'][name] = {'fp32': timed(fp32, tok), 'int8': timed(int8, tok)}
    print(name, lat['p50_ms_pool50'][name], flush=True)
json.dump(lat, open('runs/v3-latency-colab-cpu.json', 'w'), indent=2)
print(lat)

In [ ]:
import os, shutil
os.makedirs('export/runs', exist_ok=True)
for f in ['data/reranker/teacher-l6.jsonl.gz', 'data/reranker/teacher-l6.jsonl.gz.meta.json']:
    os.makedirs(os.path.dirname('export/' + f), exist_ok=True); shutil.copy(f, 'export/' + f)
for f in ['v3-student-layers.json', 'v3-selection.json', 'v3-latency-colab-cpu.json']:
    shutil.copy(f'runs/{f}', f'export/runs/{f}')
for r in ['v3-smoke', 'v3-l2', 'v3-cut']:
    os.makedirs(f'export/runs/{r}', exist_ok=True)
    for f in ['log.jsonl', 'summary.json']:
        if os.path.exists(f'runs/{r}/{f}'): shutil.copy(f'runs/{r}/{f}', f'export/runs/{r}/{f}')
for r in ['v3-l2', 'v3-cut']:
    shutil.copytree(f'runs/{r}/best', f'export/runs/{r}/best')
shutil.make_archive('sentinel-reranker-v3', 'zip', 'export')
print(os.path.getsize('sentinel-reranker-v3.zip') // 2**20, 'MB')

In [ ]:
from google.colab import files
files.download('sentinel-reranker-v3.zip')